In [1]:
!pip install fastapi uvicorn pyngrok python-multipart nest_asyncio transformers torch pillow

In [2]:
!ngrok config add-authtoken "Your token"

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from fastapi import FastAPI, UploadFile, File
from transformers import pipeline
from PIL import Image
import io



# Tải model từ Hugging Face Hub
classifier = pipeline(
    task="zero-shot-image-classification",
    model="openai/clip-vit-base-patch32",
    device=0  # dùng GPU Colab; nếu lỗi thì đổi thành -1
)


In [4]:

app = FastAPI(title="Cat vs Dog API")
CANDIDATE_LABELS = ["cat", "dog"]

@app.get("/")
def home():
    return {"message": "API is running"}

@app.get("/hi")
def sayhi():
    return {"message": "Welcom to MiniProject"}
#endpoint
@app.post("/predict")
async def predict(file: UploadFile = File(...)):

    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")

    result = classifier(
        image,
        candidate_labels=CANDIDATE_LABELS
    )

    top = result[0]

    return {
        "label": top["label"],
        "confidence": round(float(top["score"]), 4),
        "all_predictions": [
            {
                "label": x["label"],
                "score": round(float(x["score"]), 4)
            }
            for x in result
        ]
    }

In [5]:
from pyngrok import ngrok
import nest_asyncio

nest_asyncio.apply()

public_url = ngrok.connect(8000)
print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://3b21-34-105-78-109.ngrok-free.app" -> "http://localhost:8000"


In [ ]:
import uvicorn
import asyncio

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

asyncio.run(server.serve())

INFO:     Started server process [10597]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     183.81.74.94:0 - "GET / HTTP/1.1" 200 OK
INFO:     183.81.74.94:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     183.81.74.94:0 - "GET /hi HTTP/1.1" 200 OK
INFO:     183.81.74.94:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     183.81.74.94:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     183.81.74.94:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     183.81.74.94:0 - "POST /predict123123 HTTP/1.1" 404 Not Found
INFO:     183.81.74.94:0 - "GET /predict HTTP/1.1" 405 Method Not Allowed
INFO:     183.81.74.94:0 - "POST /predict123 HTTP/1.1" 404 Not Found
INFO:     183.81.74.94:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     183.81.74.94:0 - "GET / HTTP/1.1" 200 OK
INFO:     2402:800:6d73:8b8d:b415:b59c:588c:7b34:0 - "GET /hi HTTP/1.1" 200 OK
INFO:     2402:800:6d73:8b8d:b415:b59c:588c:7b34:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     2402:800:6d73:d605:7041:fac2:c050:e0e9:0 - "GET /hi HTTP/1.1" 200 OK
INFO:     2402:800:6d73:d605:7041:fac2:c050:e0e9:0 - "GET /favicon.ico H

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


INFO:     183.81.74.94:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     183.81.74.94:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     183.81.74.94:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     183.81.74.94:0 - "POST /predict HTTP/1.1" 200 OK
